[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 Easy: Kaiming Initialization

Implement **Kaiming (He) normal initialization** for weight tensors.

$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}$$

### Signature
```python
def kaiming_init(weight: Tensor) -> Tensor:
    # Initialize weight in-place with Kaiming normal
    # fan_in = weight.shape[1]
    # Returns the weight tensor
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def kaiming_init(weight):
    if weight.dim() < 2:
        raise ValueError("Kaiming initialization requires at least 2D weights.")

    # Weight shape
    # 2D (Linear): (out_features, in_features)
    # 3D (Conv1d): (out_channels, in_channels, kernel_size)
    # 4D (Conv2d): (out_channels, in_channels, kernel_height, kernel_width)
    # 5D (Conv3d): (out_channels, in_channels, kernel_depth, kernel_height, kernel_width)

    # fan_in: 출력 뉴런 하나에 연결되는 입력의 개수
    fan_in = math.prod(weight.shape[1:])
    std = math.sqrt(2 / fan_in)

    # 1) 가중치 초기화는 학습 과정이 아니므로 계산 그래프에 기록할 필요 X
    # 2) normal_()은 In-place 연산이므로 "requires_grad=True"인 학습 파라미터를 직접 수정하면 Autograd 오류가 발생
    # ------> Autograd 연산을 비활성화해야함.
    with torch.no_grad():
        weight.normal_(0, std)

    return weight

In [4]:
# 🧪 Debug
import math
w = torch.empty(256, 512)
kaiming_init(w)
print(f'Mean: {w.mean():.4f} (expect ~0)')
print(f'Std:  {w.std():.4f} (expect {math.sqrt(2/512):.4f})')

Mean: -0.0002 (expect ~0)
Std:  0.0626 (expect 0.0625)


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('weight_init')


🧪 Testing: Kaiming Initialization (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Mean approximately 0 (4.4ms)
  ✅ [2/4] Std matches sqrt(2/fan_in) (4.5ms)
  ✅ [3/4] Returns same tensor (in-place) (0.2ms)
  ✅ [4/4] Smaller fan_in gives larger std (0.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.5ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
import torch
import torch.nn as nn

# (in_features, out_features)
linear = nn.Linear(512, 256)
# W ~ U(-1/sqrt(fan_in), 1/sqrt(fan_in)) 균등분포 초기화

print(f"{linear.weight.mean():.4f}")
print(linear.weight.std())  # 1 / sqrt(3 * fan_in)
print()

# He(Kaiming) 초기화
nn.init.kaiming_normal_(
    linear.weight,
    mode="fan_in",
    nonlinearity="relu"
)

print(linear.weight.shape)
print(f"{linear.weight.mean():.4f}")
print(linear.weight.std())  # sqrt(2 / fan_in)

0.0001
tensor(0.0255, grad_fn=<StdBackward0>)

torch.Size([256, 512])
0.0001
tensor(0.0625, grad_fn=<StdBackward0>)


# Xavier Initialization

- tanh, sigmoid

In [7]:

def xavier_init(weight):
    if weight.dim() < 2:
        raise ValueError("Xavier initialization requires at least 2D weights.")

    receptive_field = math.prod(weight.shape[2:])
    fan_in = weight.shape[1] * receptive_field
    fan_out = weight.shape[0] * receptive_field

    std = math.sqrt(2 / (fan_in + fan_out))

    with torch.no_grad():
        weight.normal_(0, std)

    return weight


def xavier_uniform_init(weight):
    if weight.dim() < 2:
        raise ValueError("Xavier initialization requires at least 2D weights.")

    receptive_field = math.prod(weight.shape[2:])
    fan_in = weight.shape[1] * receptive_field
    fan_out = weight.shape[0] * receptive_field

    bound = math.sqrt(6 / (fan_in + fan_out))

    with torch.no_grad():
        weight.uniform_(-bound, bound)

    return weight